# Procesamiento de Datos — Desconectados
## HackODS UNAM 2026 · Abejas Astutas

Este notebook carga los datos crudos de las fuentes oficiales, los limpia,
los integra en un solo DataFrame por entidad federativa y exporta CSVs listos
para el dashboard de Quarto.

**Fuentes:**
- ENDUTIH 2024 — INEGI (`tr_endutih_hogares_anual_2024.csv`)
- Concentrado de pobreza 2020 — CONEVAL (`pobreza_estatal_2020.csv  (pre-convertido desde el .xlsx de CONEVAL)`)
- Estadísticas de Media Superior 2019-2024 — SEP (`MEDIA_SUPERIOR_*.csv`)

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Rutas relativas desde scripts/ al resto del repo
RAW  = "../data raw"
OUT  = "../datos"

import pathlib
pathlib.Path(OUT).mkdir(exist_ok=True)
print("Directorios listos.")


## 1 · Conectividad por entidad — ENDUTIH 2024

In [ ]:
# Cargamos solo las columnas necesarias para ahorrar memoria
hogares = pd.read_csv(
    f"{RAW}/tr_endutih_hogares_anual_2024.csv",
    encoding="latin1",
    usecols=["CVE_ENT", "P4_4", "FAC_HOG", "DOMINIO"],
)
cat_ent = pd.read_csv(f"{RAW}/tc_endutih_entidad.csv", encoding="latin1")

print(f"Hogares en muestra: {len(hogares):,}")
print(f"Distribución P4_4 (1=tiene internet, 2=no tiene):")
print(hogares["P4_4"].value_counts())


In [ ]:
# Porcentaje ponderado de hogares con internet por entidad
internet_ent = (
    hogares.groupby("CVE_ENT")
    .apply(lambda g: round(
        g.loc[g["P4_4"] == 1, "FAC_HOG"].sum() / g["FAC_HOG"].sum() * 100, 1
    ))
    .reset_index(name="internet_pct")
    .merge(cat_ent, on="CVE_ENT")
)

# Indicadores nacionales
total_fac   = hogares["FAC_HOG"].sum()
media_nac   = round(hogares.loc[hogares["P4_4"]==1, "FAC_HOG"].sum() / total_fac * 100, 1)
pct_urbano  = round(
    hogares.loc[(hogares["P4_4"]==1) & (hogares["DOMINIO"]=="U"), "FAC_HOG"].sum() /
    hogares.loc[hogares["DOMINIO"]=="U", "FAC_HOG"].sum() * 100, 1)
pct_rural   = round(
    hogares.loc[(hogares["P4_4"]==1) & (hogares["DOMINIO"]=="R"), "FAC_HOG"].sum() /
    hogares.loc[hogares["DOMINIO"]=="R", "FAC_HOG"].sum() * 100, 1)

print(f"Media nacional : {media_nac:.1f}%")
print(f"Zona urbana    : {pct_urbano:.1f}%")
print(f"Zona rural     : {pct_rural:.1f}%")
print(f"Brecha         : {pct_urbano - pct_rural:.1f} pp")
print()
print(internet_ent.sort_values("internet_pct")[["NOM_ENT","internet_pct"]].to_string())


## 2 · Pobreza por entidad — CONEVAL 2020

In [ ]:
# pobreza_estatal_2020.csv fue pre-generado desde el Excel de CONEVAL 2020
# porque openpyxl no está incluido en las dependencias del proyecto.
# El archivo original es: Concentrado_indicadores_de_pobreza_2020.xlsx
# Columnas: CVE_ENT, nom_estado, pobreza_pct_2020

pobreza = pd.read_csv(
    f"{RAW}/pobreza_estatal_2020.csv",
    encoding="utf-8",
)
pobreza = pobreza.rename(columns={"pobreza_pct_2020": "pobreza_pct"})
pobreza["CVE_ENT"] = pobreza["CVE_ENT"].astype(int)

print(f"Entidades procesadas: {len(pobreza)}")
print(pobreza.sort_values("pobreza_pct", ascending=False)[
    ["CVE_ENT","nom_estado","pobreza_pct"]].to_string())


## 3 · Abandono escolar en Media Superior — SEP 2019-2024

In [ ]:
ciclos = {
    2020: "MEDIA_SUPERIOR_2019-2020.csv",
    2021: "MEDIA_SUPERIOR_2020-2021.csv",
    2022: "MEDIA_SUPERIOR_2021-2022.csv",
    2023: "MEDIA_SUPERIOR_2022-2023.csv",
    2024: "MEDIA_SUPERIOR_2023-2024.csv",
}

abandono_hist = []  # tendencia nacional + suroeste + norte

for yr, fn in ciclos.items():
    df = pd.read_csv(
        f"{RAW}/{fn}", encoding="latin1",
        usecols=["ENTIDAD", "C_NOM_ENT", "ALUMNOS", "EXISTENTES"],
    )
    # Nacional
    nac = (df["ALUMNOS"].sum() - df["EXISTENTES"].sum()) / df["ALUMNOS"].sum() * 100
    # Suroeste: Chiapas(7), Guerrero(12), Oaxaca(20)
    sur = df[df["ENTIDAD"].isin([7, 12, 20])]
    sur_pct = (sur["ALUMNOS"].sum() - sur["EXISTENTES"].sum()) / sur["ALUMNOS"].sum() * 100
    # Norte: Baja California(2), Coahuila(5), Nuevo León(19)
    nor = df[df["ENTIDAD"].isin([2, 5, 19])]
    nor_pct = (nor["ALUMNOS"].sum() - nor["EXISTENTES"].sum()) / nor["ALUMNOS"].sum() * 100

    abandono_hist.append({
        "anio": yr,
        "Nacional": round(nac, 2),
        "Suroeste (Chis/Gro/Oax)": round(sur_pct, 2),
        "Norte (B.C./Coah/N.L.)":  round(nor_pct, 2),
    })

tendencia = pd.DataFrame(abandono_hist)
print("Tendencia de abandono escolar en Media Superior:")
print(tendencia.to_string(index=False))


In [ ]:
# Abandono por entidad — ciclo más reciente (2023-24)
ms_latest = pd.read_csv(
    f"{RAW}/MEDIA_SUPERIOR_2023-2024.csv", encoding="latin1",
    usecols=["ENTIDAD", "C_NOM_ENT", "ALUMNOS", "EXISTENTES"],
)
aban_ent = (
    ms_latest.groupby(["ENTIDAD", "C_NOM_ENT"])[["ALUMNOS", "EXISTENTES"]]
    .sum()
    .reset_index()
)
aban_ent["abandono_pct"] = (
    (aban_ent["ALUMNOS"] - aban_ent["EXISTENTES"]) / aban_ent["ALUMNOS"] * 100
).round(2)
aban_ent = aban_ent.rename(columns={"ENTIDAD": "CVE_ENT"})

print(f"Entidades: {len(aban_ent)}")
print(aban_ent[["CVE_ENT","C_NOM_ENT","ALUMNOS","EXISTENTES","abandono_pct"]]
      .sort_values("abandono_pct", ascending=False).to_string(index=False))


## 4 · Integración de fuentes

In [ ]:
# Coordenadas aproximadas de centroides por entidad (hardcoded — no cambian)
COORDS = {
    1:(22.0,-102.3), 2:(30.8,-115.6), 3:(25.1,-111.5), 4:(19.1,-90.4),
    5:(27.2,-102.1), 6:(19.1,-104.0), 7:(16.7,-92.9),  8:(28.6,-106.1),
    9:(19.4,-99.1),  10:(24.5,-104.7),11:(20.9,-101.3),12:(17.6,-99.5),
    13:(20.5,-98.9), 14:(20.7,-103.4),15:(19.4,-99.7), 16:(19.6,-101.7),
    17:(18.7,-99.0), 18:(22.0,-105.0),19:(25.6,-99.6), 20:(17.1,-96.7),
    21:(18.9,-97.7), 22:(20.6,-100.0),23:(19.2,-88.5), 24:(22.3,-101.0),
    25:(24.8,-107.4),26:(29.3,-110.7),27:(17.8,-92.7), 28:(24.3,-98.8),
    29:(19.3,-98.2), 30:(19.8,-96.9), 31:(20.7,-89.1), 32:(23.0,-102.5),
}

# Nombres cortos para labels en gráficas
NOMBRE_CORTO = {
    "Veracruz de Ignacio de la Llave": "Veracruz",
    "Michoacán de Ocampo":             "Michoacán",
    "Ciudad de México":                "CDMX",
    "Baja California Sur":             "B.C. Sur",
    "Coahuila de Zaragoza":            "Coahuila",
    "San Luis Potosí":                 "S.L. Potosí",
}

# Merge: internet (ENDUTIH) + abandono (SEP) + pobreza (CONEVAL)
estados = (
    internet_ent
    .merge(aban_ent[["CVE_ENT","abandono_pct"]], on="CVE_ENT", how="left")
    .merge(pobreza[["CVE_ENT","pobreza_pct"]],   on="CVE_ENT", how="left")
)
estados["brecha"]  = (100 - estados["internet_pct"]).round(1)
estados["nombre"]  = estados["NOM_ENT"].replace(NOMBRE_CORTO)
estados["lat"]     = estados["CVE_ENT"].map(lambda x: COORDS.get(x, (0,0))[0])
estados["lon"]     = estados["CVE_ENT"].map(lambda x: COORDS.get(x, (0,0))[1])
# Nodos solares prioritarios: estados críticos del suroeste
estados["nodo"]    = estados["CVE_ENT"].isin([7, 12, 20, 30]).astype(int)

print(f"DataFrame final: {estados.shape}")
print(estados.sort_values("internet_pct")[
    ["CVE_ENT","nombre","internet_pct","pobreza_pct","abandono_pct","brecha"]
].to_string(index=False))


## 5 · Exportar CSVs procesados

In [ ]:
# CSV principal — un registro por entidad federativa
cols_export = [
    "CVE_ENT","NOM_ENT","nombre",
    "internet_pct","pobreza_pct","abandono_pct","brecha",
    "lat","lon","nodo",
]
estados[cols_export].to_csv(f"{OUT}/estados.csv", index=False, encoding="utf-8")
print(f"✔  datos/estados.csv ({len(estados)} filas)")

# Indicadores nacionales — fila única
pd.DataFrame([{
    "media_nacional": media_nac,
    "pct_urbano":     pct_urbano,
    "pct_rural":      pct_rural,
    "brecha_ur":      round(pct_urbano - pct_rural, 1),
}]).to_csv(f"{OUT}/nacional.csv", index=False, encoding="utf-8")
print("✔  datos/nacional.csv")

# Tendencia histórica de abandono
tendencia.to_csv(f"{OUT}/tendencia_abandono.csv", index=False, encoding="utf-8")
print(f"✔  datos/tendencia_abandono.csv ({len(tendencia)} filas)")

print()
print("Todos los archivos procesados exportados a datos/")


## 6 · Verificación rápida

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = "sans-serif"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: internet vs abandono
ax = axes[0]
colores = estados["internet_pct"].apply(
    lambda x: "#c0392b" if x < 65 else ("#d97706" if x < 78 else "#15803d"))
ax.scatter(estados["internet_pct"], estados["abandono_pct"],
           c=colores, s=80, alpha=0.8, edgecolors="white", linewidths=0.6)
for _, row in estados.iterrows():
    ax.annotate(row["nombre"], (row["internet_pct"], row["abandono_pct"]),
                fontsize=6, ha="center", va="bottom", color="#374151")
ax.set_xlabel("Hogares con internet (%) — ENDUTIH 2024")
ax.set_ylabel("Abandono escolar MS (%) — SEP 2023-24")
ax.set_title("Correlación: conectividad vs abandono escolar", fontsize=11, fontweight="bold")
ax.axvline(media_nac, color="#1d4ed8", linestyle="--", linewidth=1, alpha=0.6)
ax.text(media_nac+0.5, ax.get_ylim()[1]*0.97, f"Media Mex. {media_nac}%",
        color="#1d4ed8", fontsize=8)
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.set_facecolor("#f8fafc")

# Tendencia abandono
ax2 = axes[1]
colores_t = {"Nacional":"#1d4ed8","Suroeste (Chis/Gro/Oax)":"#c0392b",
             "Norte (B.C./Coah/N.L.)":"#15803d"}
for serie, color in colores_t.items():
    ax2.plot(tendencia["anio"], tendencia[serie], "o-",
             color=color, linewidth=2, markersize=6, label=serie)
ax2.set_xlabel("Ciclo escolar (año de término)")
ax2.set_ylabel("Tasa de abandono escolar (%)")
ax2.set_title("Tendencia de abandono escolar MS 2019-2024", fontsize=11, fontweight="bold")
ax2.legend(fontsize=8)
ax2.set_xticks([2020,2021,2022,2023,2024])
ax2.set_xticklabels(["2019-20","2020-21","2021-22","2022-23","2023-24"], rotation=20)
ax2.grid(True, alpha=0.3, linewidth=0.5)
ax2.set_facecolor("#f8fafc")

plt.tight_layout()
plt.savefig(f"{OUT}/verificacion_datos.png", dpi=120, bbox_inches="tight")
plt.show()
print("✔  Verificación visual guardada en datos/verificacion_datos.png")
